# Ανάλυση και πρόβλεψη μετρήσεων TelosB
Το παρεχόμενο `synthetic-3days.csv` επιτρέπει άμεση εκτέλεση του notebook. Για πραγματικά δεδομένα: `python3 host/export_csv.py --output analysis/measurements.csv`. Αν υπάρχει `measurements.csv`, αυτό έχει προτεραιότητα. Η προέλευση δηλώνεται στο CSV. Οι συνθετικές μετρήσεις δοκιμάζουν τη διαδικασία, αλλά δεν αποτελούν πραγματική συλλογή τριών ημερών. Τα μοντέλα προβλέπουν την επόμενη τιμή ενός κόμβου με χρονολογικό διαχωρισμό εκπαίδευσης/ελέγχου.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA

DATA = Path('measurements.csv')
if not DATA.exists():
    DATA = Path('synthetic-3days.csv')
SOURCE_ID = 1
TARGET = 'temperature_c'  # ή 'humidity_pct'
if not DATA.exists():
    raise FileNotFoundError(f'Λείπει το {DATA}. Εκτελέστε πρώτα την εξαγωγή από MongoDB.')
df = pd.read_csv(DATA)
required = {'timestamp', 'source_id', 'count', 'temperature_c', 'humidity_pct'}
if not required.issubset(df.columns):
    raise ValueError(f'Λείπουν στήλες: {required - set(df.columns)}')
df['timestamp'] = pd.to_datetime(df['timestamp'].map(pd.Timestamp), utc=True)
df = df.sort_values('timestamp')
origins = sorted(df['data_origin'].fillna('real').unique()) if 'data_origin' in df else ['real']
print('Προέλευση δεδομένων:', ', '.join(origins))
if 'synthetic' in origins:
    print('ΠΡΟΣΟΧΗ: οι τιμές είναι προσομοιωμένες και δεν μετρήθηκαν από mote.')
display(df.groupby('source_id').agg(samples=('count', 'size'), first=('timestamp', 'min'), last=('timestamp', 'max')))


In [ ]:
mote = df.loc[df['source_id'] == SOURCE_ID].copy()
if mote.empty:
    raise ValueError(f'Δεν υπάρχουν μετρήσεις για κόμβο {SOURCE_ID}')
print('Διάρκεια:', mote['timestamp'].max() - mote['timestamp'].min())
display(mote[['temperature_c', 'humidity_pct', 'photo_raw', 'solar_raw']].describe())
display(mote.isna().sum().to_frame('missing'))
mote.plot(x='timestamp', y=['temperature_c', 'humidity_pct'], subplots=True, figsize=(12, 6), title=f'Κόμβος {SOURCE_ID}')
plt.tight_layout()
plt.show()


## Προεπεξεργασία
Ενοποιούμε τις λήψεις σε βήματα 20 δευτερολέπτων. Συμπληρώνουμε μόνο κενά έως δύο συνεχόμενων βημάτων, ώστε μεγάλα διαστήματα χωρίς λήψη να μην κατασκευάζουν τεχνητά δεδομένα. Η αξιολόγηση γίνεται στο τελευταίο 20% του χρόνου.

In [ ]:
series = mote.set_index('timestamp')[TARGET].astype(float).resample('20s').mean()
series = series.interpolate(method='time', limit=2).dropna()
if len(series) < 30:
    raise ValueError(f'Μόνο {len(series)} σημεία. Συλλέξτε περισσότερες πραγματικές μετρήσεις πριν από την εκπαίδευση.')
features = pd.DataFrame({'current': series, 'lag_1': series.shift(1), 'lag_2': series.shift(2), 'next_value': series.shift(-1)}).dropna()
split = int(len(features) * 0.8)
train, test = features.iloc[:split], features.iloc[split:]
X_columns = ['current', 'lag_1', 'lag_2']
linear = LinearRegression().fit(train[X_columns], train['next_value'])
linear_pred = linear.predict(test[X_columns])
print('Linear Regression MAE:', mean_absolute_error(test['next_value'], linear_pred))
print('Linear Regression RMSE:', np.sqrt(mean_squared_error(test['next_value'], linear_pred)))


In [ ]:
# ARIMA προβλέπει από το αρχικό 80% χωρίς να βλέπει τις μελλοντικές τιμές.
cutoff = int(len(series) * 0.8)
arima_train, arima_test = series.iloc[:cutoff], series.iloc[cutoff:]
arima = ARIMA(arima_train.to_numpy(), order=(1, 0, 0)).fit()
arima_pred = arima.forecast(steps=len(arima_test))
print('ARIMA MAE:', mean_absolute_error(arima_test, arima_pred))
print('ARIMA RMSE:', np.sqrt(mean_squared_error(arima_test, arima_pred)))
ax = arima_test.plot(figsize=(12, 4), label='Πραγματικό')
pd.Series(arima_pred, index=arima_test.index).plot(ax=ax, label='ARIMA')
ax.legend(); ax.set_ylabel(TARGET); plt.show()
